# Group 3, Notebook_A (Student A): data, EDA, cleaning, SQL, RQ1 figures

**Topic**: Hourly solar power forecasting (NREL)

**Role**: Student A handles Steps 0 to 3 and the RQ1 figures part of Step 5, and hands off `data/processed/feat.parquet` together with `manifest.json` to Student B (Notebook_B). Run sequentially in VS Code with the Python and Jupyter extensions, interpreter `.venv`. Each step has a guidance cell (*what to do, why, how to read it*) followed by a code cell; every table is saved to `report/` as `table_*.csv`, every figure as `fig_*.png`.

**Rules**: seed 42; never delete rows without recording a reason; every AI prompt is logged in your personal Audit Log; report at slot 1 on Tuesday and Friday using the template in the last cell.

## Step 0: environment and project folder structure

Run once in the VS Code Terminal, not in a Python cell. After installation, select the `.venv` interpreter (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [2]:
# Environment check and shared imports for the whole notebook
import pandas as pd, glob, os 
import numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
from ydata_profiling import ProfileReport
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

conn = duckdb.connect()

print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


C:\Users\Ayufuyu\AppData\Local\Temp\ipykernel_26524\842659584.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## Step 1: Pick the random 20 plants based on seed=42 and load it into the DataFrame

Pick random 20 plants on seed 42

In [4]:
# Step 1b: load 20 NREL plants (Actual_*.csv, 5-minute) and aggregate to hourly
all_files = glob.glob('data/raw/Actual_*.csv')
np.random.seed(42)

files = np.random.choice(all_files, 20, replace=False).tolist()

frames = []
for f in files:
    d = pd.read_csv(f, parse_dates=['LocalTime']); d['plant'] = os.path.basename(f)[:30]
    # Aggregate the data to be hourly instead of 5-minute batches, using the mean of 12 5-minute batches for 1 hour
    frames.append(d.set_index('LocalTime').resample('h')['Power(MW)'].mean().rename('power').reset_index().assign(plant=d.plant.iloc[0]))
df = pd.concat(frames)

# Find the row and column count, then how many unique plant found (should be 20)
print(df.shape, df.plant.nunique())
# Make a new column for the maximum power a plant outputted (separated from df)
cap = df.groupby('plant').power.max().rename('cap_mw')

(175200, 3) 20


### Automatic profiling

`ydata-profiling` on a sample of 20,000 rows generates `report/profile.html`; open it in a browser for a quick look at distributions, missingness, and correlations before writing your own EDA in the following cells.

In [5]:
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/dataset1_profiling.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 425.52it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### Understand the data

In [6]:
print(df.head(16))
df.shape
df.info()
df.describe()
df.columns
df.dtypes
df.nunique()

             LocalTime     power                           plant
0  2006-01-01 00:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
1  2006-01-01 01:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
2  2006-01-01 02:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
3  2006-01-01 03:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
4  2006-01-01 04:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
5  2006-01-01 05:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
6  2006-01-01 06:00:00  0.000000  Actual_33.65_-117.25_2006_DPV_
7  2006-01-01 07:00:00  0.066667  Actual_33.65_-117.25_2006_DPV_
8  2006-01-01 08:00:00  5.266667  Actual_33.65_-117.25_2006_DPV_
9  2006-01-01 09:00:00  8.341667  Actual_33.65_-117.25_2006_DPV_
10 2006-01-01 10:00:00  8.391667  Actual_33.65_-117.25_2006_DPV_
11 2006-01-01 11:00:00  7.733333  Actual_33.65_-117.25_2006_DPV_
12 2006-01-01 12:00:00  8.133333  Actual_33.65_-117.25_2006_DPV_
13 2006-01-01 13:00:00  3.075000  Actual_33.65_-117.25_2006_DPV_
14 2006-01-01 14:00:00  3

LocalTime     8760
power        19409
plant           20
dtype: int64

### Check the second data source (weather data)

In [7]:
df2 = pd.read_csv('data/raw/nsrdb_ca_2006.csv', parse_dates=['ts'])

sample = df2.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/dataset2_profiling.html')
df2.describe(include='all').T.to_csv('report/table_describe_raw2.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 500.01it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
print(df2.head(16))
df2.shape
df2.info()
df2.describe()
df2.columns
df2.dtypes
df2.nunique()

    site_id                  ts  GHI  DNI  cloud  temp
0         0 2006-01-01 00:00:00    0    0      1   8.6
1         0 2006-01-01 01:00:00    0    0      7   8.7
2         0 2006-01-01 02:00:00    0    0      7   8.8
3         0 2006-01-01 03:00:00    0    0      7   8.8
4         0 2006-01-01 04:00:00    0    0      7   8.8
5         0 2006-01-01 05:00:00    0    0      7   8.7
6         0 2006-01-01 06:00:00    0    0      7   8.7
7         0 2006-01-01 07:00:00    0    0      7   9.0
8         0 2006-01-01 08:00:00    6    0      7   9.4
9         0 2006-01-01 09:00:00   77    4      7   9.5
10        0 2006-01-01 10:00:00    6    0      8   9.4
11        0 2006-01-01 11:00:00   46    0      7   9.3
12        0 2006-01-01 12:00:00  136   10      7   9.0
13        0 2006-01-01 13:00:00   19    0      7   8.5
14        0 2006-01-01 14:00:00   52    0      7   8.1
15        0 2006-01-01 15:00:00   18    0      6   7.6
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172459 entries,

site_id      20
ts         8760
GHI        1078
DNI        1029
cloud        10
temp        522
dtype: int64

## Step 2: understand the data, clean it, and merge in a second source

**What to do**: standardize column names; drop duplicates by key; drop missing values in the target variable; remove values outside the physically valid range per agency documentation; build a complete time key and interpolate short gaps; merge in the second data source; save to parquet.

**Check**: row count after the merge doesn't increase; match rate above 90%; log the number of rows dropped at each step in the cleaning log (next cell).

In [9]:
# Builds plant_locations.csv
import re
rows = []
for f in files:
    name = f.split('\\')[-1] if '\\' in f else f.split('/')[-1]
    lat, lon = re.findall(r'Actual_(-?\d+\.\d+)_(-?\d+\.\d+)', name)[0]
    rows.append({'plant': name[:30], 'lat': float(lat), 'lon': float(lon)})
pd.DataFrame(rows).to_csv('data/raw/plant_locations.csv', index=False)
print('Saved', len(rows), 'plant locations')

Saved 20 plant locations


In [10]:
import glob, pandas as pd

files = sorted(glob.glob('data/raw/nsrdb_downloads/*.csv'))
plants = pd.read_csv('data/raw/plant_locations.csv')

all_data, sites = [], []
for i, f in enumerate(files):
    meta = pd.read_csv(f, nrows=1)              # first row = metadata (lat, lon, etc.)
    d = pd.read_csv(f, skiprows=2)               # actual data starts after 2 header rows
    d['site_id'] = i
    d['ts'] = pd.to_datetime(d[['Year','Month','Day','Hour']])
    d = d.rename(columns={'Cloud Type':'cloud', 'Temperature':'temp'})
    all_data.append(d[['site_id','ts','GHI','DNI','cloud','temp']])
    sites.append({'site_id': i, 'lat': meta['Latitude'][0], 'lon': meta['Longitude'][0]})

pd.concat(all_data).to_csv('data/raw/nsrdb_ca_2006.csv', index=False)
pd.DataFrame(sites).to_csv('data/raw/nsrdb_sites.csv', index=False)
print('Combined', len(files), 'sites into nsrdb_ca_2006.csv')

Combined 20 sites into nsrdb_ca_2006.csv


In [11]:
# Step 2: drop night hours, merge NSRDB (GHI, DNI, cloud) by nearest coordinates
loc = pd.read_csv(
    'data/raw/plant_locations.csv'
)  # plant, lat, lon

nsr_loc = pd.read_csv(
    'data/raw/nsrdb_sites.csv'
)  # site_id, lat, lon

from sklearn.neighbors import BallTree
import numpy as np

# Find the nearest NSRDB site to each solar plant
tree = BallTree(
    np.radians(nsr_loc[['lat', 'lon']]),
    metric='haversine'
)

_, i = tree.query(
    np.radians(loc[['lat', 'lon']]),
    k=1
)

loc['site_id'] = nsr_loc.site_id.values[i[:, 0]]

# Merge plant information + capacity + NSRDB data
m = (
    df
    .merge(loc[['plant', 'site_id']], on='plant')
    .merge(cap, on='plant')
    .merge(
        df2,
        left_on=['site_id', 'LocalTime'],
        right_on=['site_id', 'ts'],
        how='left'
    )
)

print(
    'share with NSRDB data:',
    m['GHI'].notna().mean().round(3)
)

# only keep entries where GHI > 0
m = m[m.GHI > 0].dropna(subset=['power'])
# Calculate capacity factor
m['cf'] = m.power / m.cap_mw

share with NSRDB data: 1.0


In [12]:
# Check that row count after the merge doesn't increase
print('df rows:', len(df))
print('m rows (before GHI/power filter):', len(df.merge(loc[['plant','site_id']], on='plant').merge(cap, on='plant').merge(df2, left_on=['site_id','LocalTime'], right_on=['site_id','ts'], how='left')))

df rows: 175200
m rows (before GHI/power filter): 175200


### 2.1. Cleaning log (table to include in the paper)

For each step: step name, rows before, rows after, rows dropped, documented reason.

In [13]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); 
        d = fn(d); 
        rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])

_, cleaning_log = clean_log(m, [
    ('drop duplicates', lambda x: x.drop_duplicates(['plant', 'ts']), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['cf']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates        87302       87302        0  same unit and timestamp
1  drop missing target        87302       87302        0       cannot be forecast


In [14]:
m.info()
m.head()

<class 'pandas.core.frame.DataFrame'>
Index: 87302 entries, 7 to 175193
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   LocalTime  87302 non-null  datetime64[ns]
 1   power      87302 non-null  float64       
 2   plant      87302 non-null  object        
 3   site_id    87302 non-null  int64         
 4   cap_mw     87302 non-null  float64       
 5   ts         87302 non-null  datetime64[ns]
 6   GHI        87302 non-null  int64         
 7   DNI        87302 non-null  int64         
 8   cloud      87302 non-null  int64         
 9   temp       87302 non-null  float64       
 10  cf         87302 non-null  float64       
dtypes: datetime64[ns](2), float64(4), int64(4), object(1)
memory usage: 8.0+ MB


,LocalTime,power,plant,site_id,cap_mw,ts,GHI,DNI,cloud,temp,cf
7,2006-01-01 07:00:00,0.066667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,2006-01-01 07:00:00,75,433,0,10.6,0.002477
8,2006-01-01 08:00:00,5.266667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,2006-01-01 08:00:00,163,99,7,11.8,0.195666
9,2006-01-01 09:00:00,8.341667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,2006-01-01 09:00:00,155,14,8,12.9,0.309907
10,2006-01-01 10:00:00,8.391667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,2006-01-01 10:00:00,217,21,7,14.1,0.311765
11,2006-01-01 11:00:00,7.733333,Actual_33.65_-117.25_2006_DPV_,15,26.916667,2006-01-01 11:00:00,218,18,7,15.3,0.287307


In [ ]:
# Drop the duplicated time column (Localtime and ts)
m = m.drop(['ts'], axis=1).rename(columns = {'LocalTime': "time"})
m.head()

,time,power,plant,site_id,cap_mw,GHI,DNI,cloud,temp,cf
7,2006-01-01 07:00:00,0.066667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,75,433,0,10.6,0.002477
8,2006-01-01 08:00:00,5.266667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,163,99,7,11.8,0.195666
9,2006-01-01 09:00:00,8.341667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,155,14,8,12.9,0.309907
10,2006-01-01 10:00:00,8.391667,Actual_33.65_-117.25_2006_DPV_,15,26.916667,217,21,7,14.1,0.311765
11,2006-01-01 11:00:00,7.733333,Actual_33.65_-117.25_2006_DPV_,15,26.916667,218,18,7,15.3,0.287307


## Publish the data frame to a parquet file so other notebooks can retrieve it

In [21]:
m.reset_index(drop=True).to_parquet(
    'data/processed/solar.parquet',
    index=False
)